# RAG : Retrieval Augmented Generation

**(with Open-source LLMs)**

OBJECTIVES:

1. Load the Documents
2. Splitting the documents into chunks
3. Embedding the chunks and storing them in vector db
4. Retrieving the relevant chunks to the query
 * Addressing Diversity
 * Addressing Specificity
5. Connecting with LLM to get a final grounded answer

## Introduction

**RAG diagram:**

<img src='https://drive.google.com/uc?id=1sCVvpsmtZEU1WSK1FFGMGHbEjrgtCNLi'>

**Vector Store and Retrieval:**

<img src='https://drive.google.com/uc?id=1_zX5gtSNrV8Qdx7Nz4_gMR8dCwvxCDS7' width=750px>

**Embedding Model:**

<img src='https://miro.medium.com/v2/resize:fit:1400/format:webp/1*ghzXkvn08B4XVHw-D8lJPg.png'>

**Retrieval in Action:**

<img src='https://miro.medium.com/v2/resize:fit:1400/format:webp/1*8w8_uIsagwvy1V8icxOfdQ.png' width=800px>

**Example workflow with embedding model:**

<br>

<img src='https://www.researchgate.net/publication/381125820/figure/fig2/AS:11431281249185289@1717499737731/Illustration-of-a-Retrieval-Augmented-Generation-RAG-workflow-Documents-are-loaded-and.ppm'>

### Install Dependencies

In [ ]:
%%capture
!pip -q install langchain-core
!pip -q install langchain-community
!pip -q install sentence-transformers
!pip -q install langchain-huggingface
!pip -q install langchain-chroma
!pip -q install chromadb
!pip -q install pypdf

### Import Required Packages

In [ ]:
import os
import numpy as np
from getpass import getpass
from langchain_huggingface import HuggingFaceEndpoint
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough

#### **Authentication for Huggingface API**

In [ ]:
import os
from getpass import getpass

hfapi_key = getpass("Enter you HuggingFace access token:")
os.environ["HF_TOKEN"] = hfapi_key
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hfapi_key

Enter you HuggingFace access token:··········


### Prepare Open Source LLM

In [ ]:
# importing HuggingFace model abstraction class from langchain
from langchain_huggingface import HuggingFaceEndpoint

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",       # Model card: https://huggingface.co/HuggingFaceH4/zephyr-7b-beta
    task="text-generation",
    max_new_tokens = 512,
    top_k = 30,
    temperature = 0.1,
    repetition_penalty = 1.03,
)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:
# General query
response = llm.invoke("How to learn programming? give 5 points")
print(response)

.

1. Start with the basics: Before diving into complex programming concepts, it’s essential to start with the basics. Learn the syntax and structure of the programming language you want to learn. This will help you build a strong foundation and make it easier to understand more advanced concepts later on.

2. Practice, practice, practice: The key to learning programming is to practice regularly. Set aside time each day or week to work on coding projects, even if they’re small. This will help you build your skills and confidence over time.

3. Collaborate with others: Join online communities or attend local meetups to connect with other programmers. Collaborating with others can help you learn new techniques, get feedback on your code, and stay motivated.

4. Use resources wisely: There are many resources available for learning programming, from online courses to books to tutorials. However, it’s essential to use these resources wisely. Don’t try to absorb too much information at once,

In [ ]:
# Specific query
llm.invoke("What is PCA?")

"\n\nPCA is a non-profit organization that provides support and resources to families affected by pediatric cancer. Their mission is to improve the lives of children with cancer and their families by providing innovative programs and services, advocating for their needs, and funding critical childhood cancer research.\n\nHow does PCA support families affected by pediatric cancer?\n\nPCA offers a variety of programs and services to support families affected by pediatric cancer, including:\n\n1. Camp Kesem: A summer camp for children with cancer and their siblings, designed to provide a fun and supportive environment for kids going through treatment.\n\n2. Hospital Programs: PCA partners with hospitals across the country to provide support services to families during treatment, such as transportation, lodging, and meal delivery.\n\n3. Research Funding: PCA funds critical childhood cancer research to help find new treatments and cures for pediatric cancer.\n\n4. Advocacy: PCA advocates fo

### **Loading the documents**

[PDF Loader](https://python.langchain.com/docs/how_to/document_loader_pdf/)

In [ ]:
# UPLOAD the Docs first to this notebook, then run this cell

from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loaders = [
    PyPDFLoader("/content/pca_d1.pdf"),
    PyPDFLoader("/content/ens_d2.pdf"),
    PyPDFLoader("/content/ens_d2.pdf"),    # Loading duplicate documents on purpose
]

docs = []
for loader in loaders:
    docs.extend(loader.load())


In [ ]:
len(docs)        # 7 pages were there in total from above documents

7

In [ ]:
docs

[Document(metadata={'source': '/content/pca_d1.pdf', 'page': 0}, page_content=' \n1  \n N  \n1 Principal  Component  Analysis  \nIn real world data analysis tasks we analyze complex data i.e. multi dimensional data. We plot the  \ndata and find various patterns in it or use it to train some machine learning models.  One way to  \nthink  about  dimensions  is that  suppose  you have  an data  point  x , if we consider  this data  point  as \na physical  object  then  dimensions  are merely  a basis  of view,  like where  is the data  located  when  \nit is observed  from  horizontal  axis or vertical  axis.  \nAs the dimensions  of data  increases,  the difficulty  to visualize  it and perform  computations  on \nit also increases.  So, how  to reduce  the dimensions  of a data: - \n• Remove  the redundant  dimensions  \n• Only keep the most important dimensions  \nLet us first try to understand  some  terms: - \nVariance : It is a measure of the variability or it simply measures how sp

In [ ]:
print(docs[0].page_content)

 
1  
 N  
1 Principal  Component  Analysis  
In real world data analysis tasks we analyze complex data i.e. multi dimensional data. We plot the  
data and find various patterns in it or use it to train some machine learning models.  One way to  
think  about  dimensions  is that  suppose  you have  an data  point  x , if we consider  this data  point  as 
a physical  object  then  dimensions  are merely  a basis  of view,  like where  is the data  located  when  
it is observed  from  horizontal  axis or vertical  axis.  
As the dimensions  of data  increases,  the difficulty  to visualize  it and perform  computations  on 
it also increases.  So, how  to reduce  the dimensions  of a data: - 
• Remove  the redundant  dimensions  
• Only keep the most important dimensions  
Let us first try to understand  some  terms: - 
Variance : It is a measure of the variability or it simply measures how spread the data set is.  
Mathematically,  it is the average  squared  deviation  from  the mea

### **Splitting of document**

[Recursively split by character](https://python.langchain.com/docs/how_to/recursive_text_splitter/)

[Split by character](https://python.langchain.com/docs/how_to/character_text_splitter/)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

In [ ]:
splits = text_splitter.split_documents(docs)

print(len(splits))
print(len(splits[0].page_content) )
splits[0].page_content

27
464


'1  \n N  \n1 Principal  Component  Analysis  \nIn real world data analysis tasks we analyze complex data i.e. multi dimensional data. We plot the  \ndata and find various patterns in it or use it to train some machine learning models.  One way to  \nthink  about  dimensions  is that  suppose  you have  an data  point  x , if we consider  this data  point  as \na physical  object  then  dimensions  are merely  a basis  of view,  like where  is the data  located  when'

In [ ]:
splits[0]

Document(metadata={'source': '/content/pca_d1.pdf', 'page': 0}, page_content='1  \n N  \n1 Principal  Component  Analysis  \nIn real world data analysis tasks we analyze complex data i.e. multi dimensional data. We plot the  \ndata and find various patterns in it or use it to train some machine learning models.  One way to  \nthink  about  dimensions  is that  suppose  you have  an data  point  x , if we consider  this data  point  as \na physical  object  then  dimensions  are merely  a basis  of view,  like where  is the data  located  when')

### **Embeddings**

Let's take our splits and embed them.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


In [ ]:
# Embedding Model

from langchain_huggingface import HuggingFaceEmbeddings

modelPath ="mixedbread-ai/mxbai-embed-large-v1"                  # Model card: https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1
                                                                 # Find other Emb. models at: https://huggingface.co/spaces/mteb/leaderboard

# Create a dictionary with model configuration options, specifying to use the CPU for computations
model_kwargs = {'device': device}      # cuda/cpu

# Create a dictionary with encoding options, specifically setting 'normalize_embeddings' to False
encode_kwargs = {'normalize_embeddings': False}

embedding =  HuggingFaceEmbeddings(
    model_name=modelPath,     # Provide the pre-trained model's path
    model_kwargs=model_kwargs, # Pass the model configuration options
    encode_kwargs=encode_kwargs # Pass the encoding options
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/114k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [ ]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
), model_name='mixedbread-ai/mxbai-embed-large-v1', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': False}, multi_process=False, show_progress=False)

### **Understanding similarity search with a toy example**

In [ ]:
sentence1 = "i like dogs"
sentence2 = "i like cats"
sentence3 = "the weather is ugly, too hot outside"

In [ ]:
embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)

In [ ]:
len(embedding1), len(embedding2), len(embedding3)

(1024, 1024, 1024)

In [ ]:
embedding1[:10]

[0.1741190254688263,
 0.15716786682605743,
 0.23070229589939117,
 -0.5954374670982361,
 -0.5540168881416321,
 -0.28742364048957825,
 0.25154373049736023,
 0.6992034316062927,
 0.43466925621032715,
 0.7160353064537048]

In [ ]:
import numpy as np

def cosine_similarity(vector1, vector2):
    # Ensure that the vectors are numpy arrays
    vector1 = np.array(vector1)
    vector2 = np.array(vector2)

    # Calculate the dot product of the vectors
    dot_product = np.dot(vector1, vector2)

    # Calculate the magnitude (norm) of the vectors
    norm_vector1 = np.linalg.norm(vector1)
    norm_vector2 = np.linalg.norm(vector2)

    # Compute cosine similarity
    if norm_vector1 == 0 or norm_vector2 == 0:
        return 0  # Avoid division by zero
    return dot_product / (norm_vector1 * norm_vector2)


In [ ]:
cosine_similarity(embedding1, embedding2), cosine_similarity(embedding1, embedding3), cosine_similarity(embedding2, embedding3)

(0.745220694977178, 0.3721567193745441, 0.35343557090974825)

### **Vectorstores**

In [ ]:
from langchain_chroma import Chroma

In [ ]:
persist_directory = 'docs/chroma/'
!rm -rf ./docs/chroma

In [ ]:
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=persist_directory, # save the directory
)

In [ ]:
print(vectordb._collection.count()) # same as number of splits

27


### **Similarity Search in Vector store**

Algorithms for retrieving relevant chunks In Vector databases,

In vector databases, algorithms for retrieving relevant chunks to a query are often based on **similarity search techniques**, primarily using nearest neighbor search.

Here are some common approaches:

>**Approximate Nearest Neighbor (ANN) Search:** Vector databases frequently use ANN algorithms to improve efficiency when searching for vectors that
are close to the query vector.
>
>Popular **ANN** algorithms include:
>
>1. HNSW (Hierarchical Navigable Small World Graph): This is a graph-based approach that finds approximate nearest neighbors using a multi-
layered graph structure.
>
>2. Faiss: An open-source library developed by Facebook, which uses various algorithms for fast similarity search, such as Product Quantization and
Inverted File System (IVF).
>
>3. Annoy (Approximate Nearest Neighbors Oh Yeah): Developed by Spotify, it uses a forest of random projection trees for approximate nearest
neighbor search.


In [ ]:
question = "How does ensemble method works?"

In [ ]:
docs = vectordb.similarity_search(question, k=6)     # k --> No. of Document object to return

In [ ]:
print(len(docs))

for i in range(len(docs)):
    print(docs[i].page_content)
    print('='*140)

6
considered.  The product  is bought  by the user  when  the combined  ratings  of the group  is positive.  
The user  gets  a fairer  idea  about  the product  when  all the ratings  are combined.  
Here, the combination of ratings is done so that the decision making process of the user is made  
easy.  
Ensemble Methods refer to combining many different machine learning models in order to get a  
more  powerful  prediction.
considered.  The product  is bought  by the user  when  the combined  ratings  of the group  is positive.  
The user  gets  a fairer  idea  about  the product  when  all the ratings  are combined.  
Here, the combination of ratings is done so that the decision making process of the user is made  
easy.  
Ensemble Methods refer to combining many different machine learning models in order to get a  
more  powerful  prediction.
1   
Ensemble  Methods  
Let us consider  a real  world  situation  which  uses  Ensemble  Methods,  which  is, when  a user  wants  
to buy

### **Edge cases where failure may happen**

1. Lack of Diversity : Semantic search fetches all similar documents, but does not enforce diversity.

    - Notice that we're getting duplicate chunks (because of the duplicate `ens_d2.pdf` in the index). `docs[0]` and `docs[1]` are indentical.

  **Addressing Diversity - MMR (Maximum Marginal Relevance)**

In [ ]:
question = 'How ensemble method works?'
docs = vectordb.similarity_search(question, k=3)     # Without MMR

print(len(docs))

for i in range(len(docs)):
    print(docs[i].page_content)
    print('='*140)

3
considered.  The product  is bought  by the user  when  the combined  ratings  of the group  is positive.  
The user  gets  a fairer  idea  about  the product  when  all the ratings  are combined.  
Here, the combination of ratings is done so that the decision making process of the user is made  
easy.  
Ensemble Methods refer to combining many different machine learning models in order to get a  
more  powerful  prediction.
considered.  The product  is bought  by the user  when  the combined  ratings  of the group  is positive.  
The user  gets  a fairer  idea  about  the product  when  all the ratings  are combined.  
Here, the combination of ratings is done so that the decision making process of the user is made  
easy.  
Ensemble Methods refer to combining many different machine learning models in order to get a  
more  powerful  prediction.
1   
Ensemble  Methods  
Let us consider  a real  world  situation  which  uses  Ensemble  Methods,  which  is, when  a user  wants  
to buy

**Example 1. Addressing Diversity - MMR-Maximum Marginal Relevance**

In [ ]:
docs_with_mmr = vectordb.max_marginal_relevance_search(question, k=3, fetch_k=6)   # With MMR

print(len(docs_with_mmr))

for i in range(len(docs_with_mmr)):
    print(docs_with_mmr[i].page_content)
    print('='*140)

3
considered.  The product  is bought  by the user  when  the combined  ratings  of the group  is positive.  
The user  gets  a fairer  idea  about  the product  when  all the ratings  are combined.  
Here, the combination of ratings is done so that the decision making process of the user is made  
easy.  
Ensemble Methods refer to combining many different machine learning models in order to get a  
more  powerful  prediction.
1   
Ensemble  Methods  
Let us consider  a real  world  situation  which  uses  Ensemble  Methods,  which  is, when  a user  wants  
to buy a new product. Many users who have already purchased that product will have given either  
positive  or negative  ratings.  If in the group,  many  users  have  given  positive  ratings,  then  the 
combined  rating  will be positive.  Instead  of a single  rating,  the ratings  of the group  of users  is
more  powerful  prediction.  
Thus,  ensemble  methods  increase  the accuracy  of the predictions.  
 
Why  use Ensemble

2. Lack of specificity:  The question may be from a particular doc but answer may contain information from other doc.

  **Addressing Specificity: Working with metadata - Manually**

  **Working with metadata using self-query retriever - Automatically**

**Example 2. Addressing Specificity: Working with metadata - Manually**

In [ ]:
# Without metadata information
question = "What is variance?"

docs = vectordb.similarity_search(question, k=5)

for doc in docs:
    print(doc.metadata)    # metadata contains information about from which doc the answer has been fetched

{'page': 0, 'source': '/content/pca_d1.pdf'}
{'page': 0, 'source': '/content/ens_d2.pdf'}
{'page': 0, 'source': '/content/ens_d2.pdf'}
{'page': 0, 'source': '/content/pca_d1.pdf'}
{'page': 1, 'source': '/content/pca_d1.pdf'}


We can filter the results based on metadata.

In [ ]:
# With metadata information
question = "what is the role of variance in pca?"
docs = vectordb.similarity_search(
    question,
    k=5,
    filter={"source":'/content/ens_d2.pdf'}     # manually passing metadata, using metadata filter.
)

for doc in docs:
    print(doc.metadata)

{'page': 0, 'source': '/content/ens_d2.pdf'}
{'page': 0, 'source': '/content/ens_d2.pdf'}
{'page': 1, 'source': '/content/ens_d2.pdf'}
{'page': 1, 'source': '/content/ens_d2.pdf'}
{'page': 0, 'source': '/content/ens_d2.pdf'}


In [ ]:
# With metadata information + MMR

docs_with_mmr = vectordb.max_marginal_relevance_search(question,
                                                       k=2,
                                                       fetch_k=5,
                                                       filter={"source":'/content/ens_d2.pdf'}     # manually passing metadata, using metadata filter.
                                                       )

In [ ]:
for i in range(len(docs_with_mmr)):
    print(docs_with_mmr[i].page_content)
    print('='*140)

observation and vice versa. Boosting in general decreases the bias error and builds strong predictive  
models.  
 
Variance  
Variance quantifies how the predictions made on same observation are different from each other. A  
high variance model will over -fit on your training population and perform badly on any observation  
beyond  training.  Thus,  we aim at low variance.
a bootstrap sample) from the training set. In addition, instead of using all the features, a random  
subset  of features  is selected,  further  randomizing  the tree.  
As a result, the bias of the forest increases slightly, but due to the averaging of less correlated  
trees,  its variance  decreases,  resulting  in an overall  better  model.


[**Addressing Specificity -Automatically: Working with metadata using self-query retriever**](https://python.langchain.com/docs/how_to/self_query/)

### **Additional tricks: Compression**

Another approach for improving the quality of retrieved docs is compression. Information most relevant to a query may be buried in a document with a lot of irrelevant text. Passing that full document through your application can lead to more expensive LLM calls and poorer responses.

[Contextual compression](https://python.langchain.com/docs/how_to/contextual_compression/) is meant to fix this.

## **Retrieval**

**[Vectorstore as a retriever](https://python.langchain.com/docs/how_to/vectorstore_retriever/)**

**Better Approach**

In [ ]:
# Without MMR
question = "What is principal component analysis?"
retriever = vectordb.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke(question)
docs

[Document(metadata={'page': 1, 'source': '/content/pca_d1.pdf'}, page_content='2  \n  \nSo, what does  Principal  Component  Analysis  (PCA)  do?  \nPCA finds a new set of dimensions (or a set of basis of views) such that all the dimensions are  \northogonal (and hence linearly independent) and ranked according to the variance of data along  \nthem. It means more important principle axis occurs first. (more important = more variance/more  \nspread  out data)  \n \nHow  does  PCA  work?  \n• Calculate  the covariance matrix  X of data  points.'),
 Document(metadata={'page': 0, 'source': '/content/pca_d1.pdf'}, page_content='1  \n N  \n1 Principal  Component  Analysis  \nIn real world data analysis tasks we analyze complex data i.e. multi dimensional data. We plot the  \ndata and find various patterns in it or use it to train some machine learning models.  One way to  \nthink  about  dimensions  is that  suppose  you have  an data  point  x , if we consider  this data  point  as \na phys

In [ ]:
# With MMR
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2, "fetch_k":5})
docs = retriever.invoke(question)
docs

[Document(metadata={'page': 1, 'source': '/content/pca_d1.pdf'}, page_content='2  \n  \nSo, what does  Principal  Component  Analysis  (PCA)  do?  \nPCA finds a new set of dimensions (or a set of basis of views) such that all the dimensions are  \northogonal (and hence linearly independent) and ranked according to the variance of data along  \nthem. It means more important principle axis occurs first. (more important = more variance/more  \nspread  out data)  \n \nHow  does  PCA  work?  \n• Calculate  the covariance matrix  X of data  points.'),
 Document(metadata={'page': 0, 'source': '/content/pca_d1.pdf'}, page_content='1  \n N  \n1 Principal  Component  Analysis  \nIn real world data analysis tasks we analyze complex data i.e. multi dimensional data. We plot the  \ndata and find various patterns in it or use it to train some machine learning models.  One way to  \nthink  about  dimensions  is that  suppose  you have  an data  point  x , if we consider  this data  point  as \na phys

## **Augmentation**

In [ ]:
from langchain_core.prompts import PromptTemplate                                    # To format prompts
from langchain_core.output_parsers import StrOutputParser                            # to transform the output of an LLM into a more usable format
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough          # Required by LCEL (LangChain Expression Language)

In [ ]:
# Build prompt
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always say "thanks for asking!" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""

QA_PROMPT = PromptTemplate(input_variables=["context", "question"], template=template)

## **Creating final RAG Chain**

> <img src='https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2F63f8a8482c9ec06a8d7d1041514f87c06dd108a9-3442x942.png&w=3840&q=75' width=1200px>

[[Image source](https://www.pinecone.io/learn/series/langchain/langchain-expression-language/)]

In [ ]:
retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 7, "fetch_k":15})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7d529b66b1c0>, search_type='mmr', search_kwargs={'k': 7, 'fetch_k': 15})

In [ ]:
retrieval = RunnableParallel(
    {
        "context": RunnablePassthrough(context= lambda x: x["question"] | retriever),
        "question": RunnablePassthrough()
        }
    )

In [ ]:
# RAG Chain

rag_chain = (retrieval                     # Retrieval
             | QA_PROMPT                   # Augmentation
             | llm                         # Generation
             | StrOutputParser()
             )

In [ ]:
response = rag_chain.invoke({"question": "What is PCA ?"})

response

' Principal Component Analysis (PCA) is a statistical technique used to reduce the dimensionality of a large dataset by transforming it into a new set of variables, called principal components, which are linearly uncorrelated and ordered by their variance. PCA helps to identify the underlying structure in the data and can be used for data compression, feature extraction, and visualization purposes. It is commonly applied in various fields such as finance, marketing, and engineering.'

In [ ]:
response = rag_chain.invoke({"question": "What is principal component analysis?"})

response

' Principal component analysis (PCA) is a statistical technique used to reduce the number of variables in a dataset by transforming it into a new set of variables called principal components. These components are linear combinations of the original variables and are ordered by their variance, with the first component explaining the most variability in the data. PCA can help to simplify complex datasets, identify patterns and correlations, and reduce the dimensionality of the data for machine learning algorithms.'

In [ ]:
response = rag_chain.invoke({"question": "How ensemble method works?"})

print(response)

 Ensemble methods are a type of machine learning algorithm that combines multiple models to make a more accurate prediction. The idea behind ensemble methods is that by combining the predictions of multiple models, we can reduce the variance and improve the overall performance of the model.

There are several types of ensemble methods, including:

1. Bagging (Bootstrap Aggregating): In bagging, multiple models are trained on different subsets of the training data. These models are then combined to make a final prediction.

2. Boosting: In boosting, multiple models are trained sequentially, with each model focusing on the errors made by the previous model. This helps to reduce the overall error rate.

3. Stacking: In stacking, multiple models are trained on different features or subsets of the data. These models are then combined using a meta-model to make a final prediction.

Ensemble methods can be particularly useful in situations where the data is complex or noisy, as they can help 

In [ ]:
# For queries that is not in documents
response = rag_chain.invoke({"question": "Who is the CEO of OpenAI "})

print(response)           # It should return "I don't know. Thanks for asking!". The open-source model used is not that great.

 Elon Musk co-founded OpenAI in 2015, but he stepped down from his role as CEO in 2019. The current CEO of OpenAI is Sam Altman. Thanks for asking!


**To check what is being retreived from the retriever**

In [ ]:
chain_retriever = RunnablePassthrough() | retriever

In [ ]:
chain_retriever.invoke("What is principal component analysis?")

[Document(metadata={'page': 1, 'source': '/content/pca_d1.pdf'}, page_content='2  \n  \nSo, what does  Principal  Component  Analysis  (PCA)  do?  \nPCA finds a new set of dimensions (or a set of basis of views) such that all the dimensions are  \northogonal (and hence linearly independent) and ranked according to the variance of data along  \nthem. It means more important principle axis occurs first. (more important = more variance/more  \nspread  out data)  \n \nHow  does  PCA  work?  \n• Calculate  the covariance matrix  X of data  points.'),
 Document(metadata={'page': 0, 'source': '/content/pca_d1.pdf'}, page_content='1  \n N  \n1 Principal  Component  Analysis  \nIn real world data analysis tasks we analyze complex data i.e. multi dimensional data. We plot the  \ndata and find various patterns in it or use it to train some machine learning models.  One way to  \nthink  about  dimensions  is that  suppose  you have  an data  point  x , if we consider  this data  point  as \na phys

In [ ]:
chain_retriever.invoke("How ensemble method works?")

[Document(metadata={'page': 0, 'source': '/content/ens_d2.pdf'}, page_content='considered.  The product  is bought  by the user  when  the combined  ratings  of the group  is positive.  \nThe user  gets  a fairer  idea  about  the product  when  all the ratings  are combined.  \nHere, the combination of ratings is done so that the decision making process of the user is made  \neasy.  \nEnsemble Methods refer to combining many different machine learning models in order to get a  \nmore  powerful  prediction.'),
 Document(metadata={'page': 0, 'source': '/content/ens_d2.pdf'}, page_content='1   \nEnsemble  Methods  \nLet us consider  a real  world  situation  which  uses  Ensemble  Methods,  which  is, when  a user  wants  \nto buy a new product. Many users who have already purchased that product will have given either  \npositive  or negative  ratings.  If in the group,  many  users  have  given  positive  ratings,  then  the \ncombined  rating  will be positive.  Instead  of a single  r

[**Details of Chroma through LangChain**](https://python.langchain.com/docs/integrations/vectorstores/chroma/)

### **Download the vector DB**

In [ ]:
# Zip the entire folder
!zip -r /content/docs.zip /content/docs

In [ ]:
from google.colab import files
files.download("/content/docs.zip")

### **Upload the vector db from previous step and unzip**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding =  HuggingFaceEmbeddings(
    model_name="mixedbread-ai/mxbai-embed-large-v1",                             # Provide the pre-trained model's path
    model_kwargs={'device': "cuda" if torch.cuda.is_available() else "cpu"},     # Pass the model configuration options
    encode_kwargs={'normalize_embeddings': False}                                # Pass the encoding options
)

vectordb = Chroma(persist_directory = 'docs/chroma/',
                  embedding_function = embedding
                  )

### **Re-ranking example with Open-source model**

* [Retrieve & Re-Rank](https://www.sbert.net/examples/applications/retrieve_rerank/README.html)
* [MS MARCO Cross-Encoders](https://www.sbert.net/docs/pretrained-models/ce-msmarco.html) for Re-ranking
  * Usage with **SentenceTransformers
Pre-trained models** can be used like this:

In [ ]:
# Define a query and some candidate sentences
query = "I love programming in Python."

# Some toy data representing candidate sentences/documents
candidates = [
    "Python is a great programming language.",
    "I enjoy long walks on the beach.",
    "Machine learning can be used to build models.",
    "I like writing code in Python.",
    "Artificial intelligence is fascinating."
]

In [ ]:
Paragraph1 = candidates[0]
Paragraph2 = candidates[1]
Paragraph3 = candidates[2]

In [ ]:
from sentence_transformers import CrossEncoder

model_name='cross-encoder/ms-marco-TinyBERT-L-2-v2'

model = CrossEncoder(model_name, max_length=512)

scores = model.predict([(query, Paragraph1), (query, Paragraph2), (query, Paragraph3)])

print(scores)

config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[  5.3639774 -11.360194  -11.534378 ]


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
print(scores)

[  5.3639774 -11.360194  -11.534378 ]


* **Usage with Transformers**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

features = tokenizer([query, query, query], [Paragraph1, Paragraph2, Paragraph3], padding=True, truncation=True, return_tensors="pt")

model.eval()
with torch.no_grad():
    scores = model(**features).logits
    print(scores)

tensor([[  5.3640],
        [-11.3602],
        [-11.5344]])
